In [ ]:
# Extract Per-Encounter Metadata

from pathlib import Path
import json
import numpy as np
import pandas as pd

HISTORY_ROOT = Path("/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/84_clean")
OUTPUT_DIR = Path("/mnt/bulk-uranus/lizhang/LiWorkSpace/data/plots/84")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USAGE_GROUPS = ["doctor", "patient", "blood_matcher", "total"]
USAGE_FIELDS = ["requests", "input_tokens", "output_tokens", "total_tokens", "cached_tokens", "reasoning_tokens"]


def parse_conversation_path(path: Path, history_root: Path) -> dict:
    rel = path.relative_to(history_root)
    run_name = rel.parts[0]          # run_1
    dataset = rel.parts[1]           # appendicitis
    filename = path.name

    prefix = f"{dataset}_"
    suffix = "_conversation_"
    if filename.startswith(prefix) and suffix in filename:
        case_id = filename[len(prefix):filename.index(suffix)]
    else:
        case_id = filename.split(suffix)[0]

    return {
        "run_id": int(run_name.replace("run_", "")),
        "dataset": dataset,
        "case_id": case_id,
        "conversation_file": str(path),
    }


def get_run_metadata(path: Path) -> dict | None:
    with path.open("r", encoding="utf-8") as f:
        conversation = json.load(f)

    for item in conversation:
        if isinstance(item, dict) and item.get("type") == "run_metadata":
            return item.get("metadata", {})
    return None


def extract_encounter_rows(history_root: Path) -> pd.DataFrame:
    rows = []

    for path in sorted(history_root.glob("run_*/*/*_conversation_*.json")):
        path_info = parse_conversation_path(path, history_root)
        metadata = get_run_metadata(path)

        if not metadata:
            rows.append({
                **path_info,
                "metadata_missing": True,
            })
            continue

        deployment = metadata.get("deployment", {})
        usage = metadata.get("token_usage", {})

        row = {
            **path_info,
            "metadata_missing": False,
            "case_start_time": metadata.get("case_start_time"),
            "case_end_time": metadata.get("case_end_time"),
            "wall_clock_seconds": metadata.get("wall_clock_seconds"),
            "doctor_model": deployment.get("doctor_model"),
            "patient_model": deployment.get("patient_model"),
            "blood_matcher_model": deployment.get("blood_matcher_model"),
        }

        for group in USAGE_GROUPS:
            group_usage = usage.get(group, {})
            for field in USAGE_FIELDS:
                row[f"{group}_{field}"] = group_usage.get(field, 0)

        wall = row.get("wall_clock_seconds")
        row["total_output_tokens_per_second"] = (
            row["total_output_tokens"] / wall if wall and wall > 0 else np.nan
        )
        row["total_tokens_per_second"] = (
            row["total_total_tokens"] / wall if wall and wall > 0 else np.nan
        )

        rows.append(row)

    return pd.DataFrame(rows)


encounters_df = extract_encounter_rows(HISTORY_ROOT)
encounters_csv = OUTPUT_DIR / "per_encounter_computational_cost_raw.csv"
encounters_df.to_csv(encounters_csv, index=False)

print(f"Saved raw per-encounter table: {encounters_csv}")
print(encounters_df.shape)
encounters_df.head()


In [ ]:
# Extended Data Table 1

def median_iqr_numeric(series: pd.Series) -> dict:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return {"median": np.nan, "q1": np.nan, "q3": np.nan, "mean": np.nan, "sd": np.nan}
    return {
        "median": s.median(),
        "q1": s.quantile(0.25),
        "q3": s.quantile(0.75),
        "mean": s.mean(),
        "sd": s.std(ddof=1),
    }


def fmt_median_iqr(series: pd.Series, digits: int = 1) -> str:
    stats = median_iqr_numeric(series)
    if pd.isna(stats["median"]):
        return "NA"
    return f"{stats['median']:.{digits}f} ({stats['q1']:.{digits}f}-{stats['q3']:.{digits}f})"


def summarize_per_encounter_cost(encounters: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    valid = encounters[~encounters["metadata_missing"].fillna(False)].copy()

    group_cols = ["doctor_model", "patient_model", "blood_matcher_model"]
    valid["blood_matcher_model"] = valid["blood_matcher_model"].fillna("None")

    numeric_metrics = [
        "wall_clock_seconds",
        "total_total_tokens",
        "total_input_tokens",
        "total_output_tokens",
        "doctor_total_tokens",
        "doctor_input_tokens",
        "doctor_output_tokens",
        "patient_total_tokens",
        "blood_matcher_total_tokens",
        "total_requests",
        "total_output_tokens_per_second",
    ]

    raw_rows = []
    display_rows = []

    for keys, group in valid.groupby(group_cols, dropna=False):
        base = dict(zip(group_cols, keys))
        base["n_encounters"] = len(group)
        base["n_cases"] = group[["dataset", "case_id"]].drop_duplicates().shape[0]

        raw_row = base.copy()
        for metric in numeric_metrics:
            stats = median_iqr_numeric(group[metric])
            for stat_name, value in stats.items():
                raw_row[f"{metric}_{stat_name}"] = value
        raw_rows.append(raw_row)

        display_rows.append({
            **base,
            "wall_clock_seconds_per_encounter_median_iqr": fmt_median_iqr(group["wall_clock_seconds"], 1),
            "total_tokens_per_encounter_median_iqr": fmt_median_iqr(group["total_total_tokens"], 0),
            "doctor_input_tokens_median_iqr": fmt_median_iqr(group["doctor_input_tokens"], 0),
            "doctor_output_tokens_median_iqr": fmt_median_iqr(group["doctor_output_tokens"], 0),
            "patient_total_tokens_median_iqr": fmt_median_iqr(group["patient_total_tokens"], 0),
            "total_requests_median_iqr": fmt_median_iqr(group["total_requests"], 0),
            "output_tokens_per_second_median_iqr": fmt_median_iqr(group["total_output_tokens_per_second"], 1),
        })

    return pd.DataFrame(raw_rows), pd.DataFrame(display_rows)


table1_raw, table1_display = summarize_per_encounter_cost(encounters_df)

table1_raw_path = OUTPUT_DIR / "table_1_per_encounter_cost_raw.csv"
table1_display_path = OUTPUT_DIR / "table_1_per_encounter_cost_display.csv"

table1_raw.to_csv(table1_raw_path, index=False)
table1_display.to_csv(table1_display_path, index=False)

print(f"Saved Table 1 raw: {table1_raw_path}")
print(f"Saved Table 1 display: {table1_display_path}")
table1_display


In [ ]:
# Extended Data Table 2

def make_case_level_consistency_cost(encounters: pd.DataFrame) -> pd.DataFrame:
    valid = encounters[~encounters["metadata_missing"].fillna(False)].copy()
    valid["blood_matcher_model"] = valid["blood_matcher_model"].fillna("None")

    group_cols = [
        "doctor_model",
        "patient_model",
        "blood_matcher_model",
        "dataset",
        "case_id",
    ]

    rows = []
    for keys, group in valid.groupby(group_cols, dropna=False):
        base = dict(zip(group_cols, keys))
        n_runs = group["run_id"].nunique()

        row = {
            **base,
            "n_runs_observed": n_runs,
            "single_run_wall_clock_seconds_mean": group["wall_clock_seconds"].mean(),
            "single_run_wall_clock_seconds_median": group["wall_clock_seconds"].median(),
            "single_run_total_tokens_mean": group["total_total_tokens"].mean(),
            "single_run_total_tokens_median": group["total_total_tokens"].median(),
            "serial_multi_run_wall_clock_seconds": group["wall_clock_seconds"].sum(),
            "parallel_multi_run_wall_clock_lower_bound_seconds": group["wall_clock_seconds"].max(),
            "multi_run_total_tokens": group["total_total_tokens"].sum(),
            "multi_run_total_requests": group["total_requests"].sum(),
        }

        row["wall_clock_overhead_vs_single_run"] = (
            row["serial_multi_run_wall_clock_seconds"] / row["single_run_wall_clock_seconds_mean"]
            if row["single_run_wall_clock_seconds_mean"] > 0 else np.nan
        )
        row["token_overhead_vs_single_run"] = (
            row["multi_run_total_tokens"] / row["single_run_total_tokens_mean"]
            if row["single_run_total_tokens_mean"] > 0 else np.nan
        )

        rows.append(row)

    return pd.DataFrame(rows)


def summarize_consistency_overhead(case_cost: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    group_cols = ["doctor_model", "patient_model", "blood_matcher_model"]
    numeric_metrics = [
        "n_runs_observed",
        "single_run_wall_clock_seconds_mean",
        "single_run_total_tokens_mean",
        "serial_multi_run_wall_clock_seconds",
        "parallel_multi_run_wall_clock_lower_bound_seconds",
        "multi_run_total_tokens",
        "multi_run_total_requests",
        "wall_clock_overhead_vs_single_run",
        "token_overhead_vs_single_run",
    ]

    raw_rows = []
    display_rows = []

    for keys, group in case_cost.groupby(group_cols, dropna=False):
        base = dict(zip(group_cols, keys))
        base["n_cases"] = len(group)
        base["n_cases_with_5_runs"] = int((group["n_runs_observed"] == 5).sum())

        raw_row = base.copy()
        for metric in numeric_metrics:
            stats = median_iqr_numeric(group[metric])
            for stat_name, value in stats.items():
                raw_row[f"{metric}_{stat_name}"] = value
        raw_rows.append(raw_row)

        display_rows.append({
            **base,
            "runs_observed_per_case_median_iqr": fmt_median_iqr(group["n_runs_observed"], 0),
            "single_run_wall_clock_seconds_median_iqr": fmt_median_iqr(group["single_run_wall_clock_seconds_mean"], 1),
            "single_run_total_tokens_median_iqr": fmt_median_iqr(group["single_run_total_tokens_mean"], 0),
            "serial_5run_wall_clock_seconds_median_iqr": fmt_median_iqr(group["serial_multi_run_wall_clock_seconds"], 1),
            "parallel_5run_wall_clock_lower_bound_seconds_median_iqr": fmt_median_iqr(group["parallel_multi_run_wall_clock_lower_bound_seconds"], 1),
            "5run_total_tokens_median_iqr": fmt_median_iqr(group["multi_run_total_tokens"], 0),
            "5run_total_requests_median_iqr": fmt_median_iqr(group["multi_run_total_requests"], 0),
            "wall_clock_overhead_vs_single_run_median_iqr": fmt_median_iqr(group["wall_clock_overhead_vs_single_run"], 1),
            "token_overhead_vs_single_run_median_iqr": fmt_median_iqr(group["token_overhead_vs_single_run"], 1),
        })

    return pd.DataFrame(raw_rows), pd.DataFrame(display_rows)


case_cost_df = make_case_level_consistency_cost(encounters_df)
table2_raw, table2_display = summarize_consistency_overhead(case_cost_df)

case_cost_path = OUTPUT_DIR / "case_level_consistency_cost_raw.csv"
table2_raw_path = OUTPUT_DIR / "table_2_consistency_overhead_raw.csv"
table2_display_path = OUTPUT_DIR / "table_2_consistency_overhead_display.csv"

case_cost_df.to_csv(case_cost_path, index=False)
table2_raw.to_csv(table2_raw_path, index=False)
table2_display.to_csv(table2_display_path, index=False)

print(f"Saved case-level cost raw: {case_cost_path}")
print(f"Saved Table 2 raw: {table2_raw_path}")
print(f"Saved Table 2 display: {table2_display_path}")
table2_display


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

GROUPS = [
    {
        "group_label": "GPT-OSS",
        "models": [
            {
                "label": "T=0.01",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/100_clean",
                "color": "#B19CCB",
            },
            {
                "label": "T=0.3",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/101_clean",
                "color": "#917BBD",
            },
            {
                "label": "T=0.6",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/99_clean",
                "color": "#6656A6",
            },
            {
                "label": "T=0.9",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/102_clean",
                "color": "#3C3B8B",
            },
        ],
    },
    {
        "group_label": "GLM-4.5-Air",
        # "plots_base_dir": "/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/aidoc_benchmark/Eval_10/GLM_4.5_Air",
        "models": [
            {
                "label": "T=0.01",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/104_clean",
                "color": "#aeecd8",
            },
            {
                "label": "T=0.3",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/105_clean",
                "color": "#7bebc6",
            },
            {
                "label": "T=0.6",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/106_clean",
                "color": "#66c2a4",
            },
            {
                "label": "T=0.9",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/107_clean",
                "color": "#08b57b",
            },
        ],
    },
    {
        "group_label": "GLM-5",
        # "plots_base_dir": "/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/aidoc_benchmark/Eval_10/GLM_5",
        "models": [
            {
                "label": "T=0.01",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/78_clean",
                "color": "#d8c49d",
            },
            {
                "label": "T=0.3",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/80_clean",
                "color": "#fed98e",
            },
            {
                "label": "T=0.6",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/79_clean",
                "color": "#c2a768",
            },
            {
                "label": "T=0.9",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/81_clean",
                "color": "#a6611a",
            },
        ],
    },
    {
        "group_label": "Qwen-3.5",
        # "plots_base_dir": "/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/aidoc_benchmark/Eval_10/Qwen3.5",
        "models": [
            {
                "label": "T=0.01",
                "history_dir": "/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/history/71_clean",
                "color": "#bdd7e7",
            },
            {
                "label": "T=0.3",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/83_clean",
                "color": "#6baed6",
            },
            {
                "label": "T=0.6",
                "history_dir": "/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/history/76_clean_5runs",
                "color": "#2171b5",
            },
            {
                "label": "T=0.9",
                "history_dir": "/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/82_clean",
                "color": "#042a4c",
            },
        ],
    },
]

OUTPUT_DIR = Path("/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USAGE_GROUPS = ["doctor", "patient", "blood_matcher", "total"]
USAGE_FIELDS = ["requests", "input_tokens", "output_tokens", "total_tokens", "cached_tokens", "reasoning_tokens"]


def parse_conversation_path(path: Path, history_root: Path) -> dict:
    rel = path.relative_to(history_root)
    run_name = rel.parts[0]
    dataset = rel.parts[1]
    filename = path.name

    prefix = f"{dataset}_"
    suffix = "_conversation_"
    if filename.startswith(prefix) and suffix in filename:
        case_id = filename[len(prefix):filename.index(suffix)]
    else:
        case_id = filename.split(suffix)[0]

    return {
        "run_id": int(run_name.replace("run_", "")),
        "dataset": dataset,
        "case_id": str(case_id),
        "conversation_file": str(path),
    }


def get_run_metadata(path: Path) -> dict | None:
    try:
        with path.open("r", encoding="utf-8") as f:
            conversation = json.load(f)
    except Exception:
        return None

    for item in conversation:
        if isinstance(item, dict) and item.get("type") == "run_metadata":
            return item.get("metadata", {})
    return None


def extract_encounters_for_experiment(group_label: str, model_cfg: dict) -> pd.DataFrame:
    history_root = Path(model_cfg["history_dir"])
    rows = []

    for path in sorted(history_root.glob("run_*/*/*_conversation_*.json")):
        path_info = parse_conversation_path(path, history_root)
        metadata = get_run_metadata(path)

        row = {
            "group_label": group_label,
            "experiment_label": model_cfg["label"],
            "history_dir": str(history_root),
            "note": model_cfg.get("note", ""),
            "color": model_cfg.get("color", ""),
            **path_info,
        }

        if not metadata:
            rows.append({
                **row,
                "metadata_missing": True,
            })
            continue

        deployment = metadata.get("deployment", {})
        usage = metadata.get("token_usage", {})

        row.update({
            "metadata_missing": False,
            "case_start_time": metadata.get("case_start_time"),
            "case_end_time": metadata.get("case_end_time"),
            "wall_clock_seconds": metadata.get("wall_clock_seconds"),
            "doctor_model": deployment.get("doctor_model"),
            "patient_model": deployment.get("patient_model"),
            "blood_matcher_model": deployment.get("blood_matcher_model"),
        })

        for group in USAGE_GROUPS:
            group_usage = usage.get(group, {})
            for field in USAGE_FIELDS:
                row[f"{group}_{field}"] = group_usage.get(field, 0)

        wall = row.get("wall_clock_seconds")
        row["total_output_tokens_per_second"] = (
            row["total_output_tokens"] / wall if wall and wall > 0 else np.nan
        )
        row["total_tokens_per_second"] = (
            row["total_total_tokens"] / wall if wall and wall > 0 else np.nan
        )

        rows.append(row)

    return pd.DataFrame(rows)


def build_all_encounters(groups: list[dict]) -> pd.DataFrame:
    frames = []
    for group in groups:
        group_label = group["group_label"]
        for model_cfg in group["models"]:
            df = extract_encounters_for_experiment(group_label, model_cfg)
            print(f"{group_label} / {model_cfg['label']}: {len(df)} encounters")
            frames.append(df)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def make_case_level_consistency_cost(encounters: pd.DataFrame) -> pd.DataFrame:
    valid = encounters[~encounters["metadata_missing"].fillna(False)].copy()
    valid["blood_matcher_model"] = valid["blood_matcher_model"].fillna("None")

    group_cols = [
        "group_label",
        "experiment_label",
        "history_dir",
        "note",
        "doctor_model",
        "patient_model",
        "blood_matcher_model",
        "dataset",
        "case_id",
    ]

    rows = []
    for keys, group in valid.groupby(group_cols, dropna=False):
        base = dict(zip(group_cols, keys))
        n_runs = group["run_id"].nunique()

        single_run_wall_mean = group["wall_clock_seconds"].mean()
        single_run_tokens_mean = group["total_total_tokens"].mean()

        row = {
            **base,
            "n_runs_observed": n_runs,
            "single_run_wall_clock_seconds_mean": single_run_wall_mean,
            "single_run_wall_clock_seconds_median": group["wall_clock_seconds"].median(),
            "single_run_total_tokens_mean": single_run_tokens_mean,
            "single_run_total_tokens_median": group["total_total_tokens"].median(),
            "serial_multi_run_wall_clock_seconds": group["wall_clock_seconds"].sum(),
            "parallel_multi_run_wall_clock_lower_bound_seconds": group["wall_clock_seconds"].max(),
            "multi_run_total_tokens": group["total_total_tokens"].sum(),
            "multi_run_total_requests": group["total_requests"].sum(),
        }

        row["wall_clock_overhead_vs_single_run"] = (
            row["serial_multi_run_wall_clock_seconds"] / single_run_wall_mean
            if single_run_wall_mean and single_run_wall_mean > 0 else np.nan
        )
        row["token_overhead_vs_single_run"] = (
            row["multi_run_total_tokens"] / single_run_tokens_mean
            if single_run_tokens_mean and single_run_tokens_mean > 0 else np.nan
        )

        rows.append(row)

    return pd.DataFrame(rows)


def median_iqr(series: pd.Series) -> dict:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return {"median": np.nan, "q1": np.nan, "q3": np.nan, "mean": np.nan, "sd": np.nan}
    return {
        "median": float(s.median()),
        "q1": float(s.quantile(0.25)),
        "q3": float(s.quantile(0.75)),
        "mean": float(s.mean()),
        "sd": float(s.std(ddof=1)),
    }


def fmt_median_iqr(series: pd.Series, digits: int = 1) -> str:
    stats = median_iqr(series)
    if pd.isna(stats["median"]):
        return "NA"
    return f"{stats['median']:.{digits}f} ({stats['q1']:.{digits}f}-{stats['q3']:.{digits}f})"


def unique_join(series: pd.Series) -> str:
    vals = sorted({str(x) for x in series.dropna().unique() if str(x) and str(x) != "nan"})
    return "; ".join(vals) if vals else "NA"


def summarize_token_request_cost(encounters: pd.DataFrame) -> pd.DataFrame:
    valid = encounters[~encounters["metadata_missing"].fillna(False)].copy()
    valid["blood_matcher_model"] = valid["blood_matcher_model"].fillna("None")

    group_cols = [
        "group_label",
        "experiment_label",
        "history_dir",
        "note",
        "doctor_model",
        "patient_model",
        "blood_matcher_model",
    ]

    rows = []
    for keys, group in valid.groupby(group_cols, dropna=False):
        base = dict(zip(group_cols, keys))
        row = {
            **base,
            "n_encounters": int(len(group)),
            "n_cases": int(group[["dataset", "case_id"]].drop_duplicates().shape[0]),
            "n_runs_observed_total": int(group["run_id"].nunique()),
            "datasets": unique_join(group["dataset"]),
            "total_tokens_per_encounter_median_iqr": fmt_median_iqr(group["total_total_tokens"], 0),
            "input_tokens_per_encounter_median_iqr": fmt_median_iqr(group["total_input_tokens"], 0),
            "output_tokens_per_encounter_median_iqr": fmt_median_iqr(group["total_output_tokens"], 0),
            "doctor_total_tokens_per_encounter_median_iqr": fmt_median_iqr(group["doctor_total_tokens"], 0),
            "doctor_input_tokens_per_encounter_median_iqr": fmt_median_iqr(group["doctor_input_tokens"], 0),
            "doctor_output_tokens_per_encounter_median_iqr": fmt_median_iqr(group["doctor_output_tokens"], 0),
            "patient_total_tokens_per_encounter_median_iqr": fmt_median_iqr(group["patient_total_tokens"], 0),
            "blood_matcher_total_tokens_per_encounter_median_iqr": fmt_median_iqr(group["blood_matcher_total_tokens"], 0),
            "requests_per_encounter_median_iqr": fmt_median_iqr(group["total_requests"], 0),
        }

        for metric in [
            "total_total_tokens",
            "total_input_tokens",
            "total_output_tokens",
            "doctor_total_tokens",
            "doctor_input_tokens",
            "doctor_output_tokens",
            "patient_total_tokens",
            "blood_matcher_total_tokens",
            "total_requests",
        ]:
            stats = median_iqr(group[metric])
            for stat_name, value in stats.items():
                row[f"{metric}_{stat_name}"] = value

        rows.append(row)

    return pd.DataFrame(rows)


def summarize_observed_latency(encounters: pd.DataFrame, case_cost: pd.DataFrame) -> pd.DataFrame:
    valid = encounters[~encounters["metadata_missing"].fillna(False)].copy()
    valid["blood_matcher_model"] = valid["blood_matcher_model"].fillna("None")

    group_cols = [
        "group_label",
        "experiment_label",
        "history_dir",
        "note",
        "doctor_model",
        "patient_model",
        "blood_matcher_model",
    ]

    rows = []
    for keys, group in valid.groupby(group_cols, dropna=False):
        base = dict(zip(group_cols, keys))

        case_subset = case_cost
        for col, value in base.items():
            case_subset = case_subset[case_subset[col] == value]

        row = {
            **base,
            "n_encounters": int(len(group)),
            "n_cases": int(case_subset.shape[0]),
            "wall_clock_seconds_per_encounter_median_iqr": fmt_median_iqr(group["wall_clock_seconds"], 1),
            "wall_clock_minutes_per_encounter_median_iqr": fmt_median_iqr(group["wall_clock_seconds"] / 60.0, 1),
            "output_tokens_per_second_median_iqr": fmt_median_iqr(group["total_output_tokens_per_second"], 1),
            "single_run_wall_clock_seconds_case_mean_median_iqr": fmt_median_iqr(case_subset["single_run_wall_clock_seconds_mean"], 1),
            "serial_5run_wall_clock_seconds_median_iqr": fmt_median_iqr(case_subset["serial_multi_run_wall_clock_seconds"], 1),
            "serial_5run_wall_clock_minutes_median_iqr": fmt_median_iqr(case_subset["serial_multi_run_wall_clock_seconds"] / 60.0, 1),
            "parallel_5run_lower_bound_seconds_median_iqr": fmt_median_iqr(case_subset["parallel_multi_run_wall_clock_lower_bound_seconds"], 1),
            "parallel_5run_lower_bound_minutes_median_iqr": fmt_median_iqr(case_subset["parallel_multi_run_wall_clock_lower_bound_seconds"] / 60.0, 1),
        }

        for metric in [
            "wall_clock_seconds",
            "total_output_tokens_per_second",
        ]:
            stats = median_iqr(group[metric])
            for stat_name, value in stats.items():
                row[f"{metric}_{stat_name}"] = value

        for metric in [
            "single_run_wall_clock_seconds_mean",
            "serial_multi_run_wall_clock_seconds",
            "parallel_multi_run_wall_clock_lower_bound_seconds",
            "multi_run_total_tokens",
            "multi_run_total_requests",
        ]:
            stats = median_iqr(case_subset[metric])
            for stat_name, value in stats.items():
                row[f"{metric}_{stat_name}"] = value

        rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
all_encounters_df = build_all_encounters(GROUPS)
all_case_cost_df = make_case_level_consistency_cost(all_encounters_df)

token_summary_df = summarize_token_request_cost(all_encounters_df)
latency_summary_df = summarize_observed_latency(all_encounters_df, all_case_cost_df)

all_encounters_path = OUTPUT_DIR / "all_per_encounter_computational_cost_raw.csv"
all_case_cost_path = OUTPUT_DIR / "all_case_level_consistency_cost_raw.csv"
token_summary_path = OUTPUT_DIR / "group_token_request_cost_summary.csv"
latency_summary_path = OUTPUT_DIR / "group_observed_latency_summary.csv"

all_encounters_df.to_csv(all_encounters_path, index=False)
all_case_cost_df.to_csv(all_case_cost_path, index=False)
token_summary_df.to_csv(token_summary_path, index=False)
latency_summary_df.to_csv(latency_summary_path, index=False)

print(f"Saved all encounter raw data: {all_encounters_path}")
print(f"Saved all case-level raw data: {all_case_cost_path}")
print(f"Saved token/request summary: {token_summary_path}")
print(f"Saved observed latency summary: {latency_summary_path}")

display(token_summary_df)
display(latency_summary_df)


In [15]:
print(
    "Median total tokens per encounter range:",
    token_summary_df["total_total_tokens_median"].min(),
    "to",
    token_summary_df["total_total_tokens_median"].max(),
)

print(
    "Median total output tokens per encounter range:",
    token_summary_df["total_output_tokens_median"].min(),
    "to",
    token_summary_df["total_output_tokens_median"].max(),
)

print(
    "Median observed wall-clock minutes per encounter range:",
    latency_summary_df["wall_clock_seconds_median"].min() / 60,
    "to",
    latency_summary_df["wall_clock_seconds_median"].max() / 60,
)

print(
    "Median 5-run total tokens per case range:",
    latency_summary_df["multi_run_total_tokens_median"].min(),
    "to",
    latency_summary_df["multi_run_total_tokens_median"].max(),
)


Median total tokens per encounter range: 121274.0 to 195431.0
Median total output tokens per encounter range: 6510.0 to 9500.0
Median observed wall-clock minutes per encounter range: 2.3825616999999997 to 6.3000473999999995
Median 5-run total tokens per case range: 614769.0 to 1031518.0


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import pandas as pd


FIGURE_OUTPUT_DIR = Path("/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary")
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def setup_style():
    try:
        arial_path = "/usr/share/fonts/truetype/msttcorefonts/Arial.ttf"
        fm.fontManager.addfont(arial_path)
        plt.rcParams["font.family"] = "Arial"
    except FileNotFoundError:
        plt.rcParams["font.family"] = "sans-serif"

    plt.rcParams["font.size"] = 6
    plt.rcParams["axes.labelsize"] = 6
    plt.rcParams["axes.titlesize"] = 7
    plt.rcParams["xtick.labelsize"] = 5
    plt.rcParams["ytick.labelsize"] = 5
    plt.rcParams["legend.fontsize"] = 5
    plt.rcParams["axes.linewidth"] = 0.5
    plt.rcParams["xtick.major.width"] = 0.5
    plt.rcParams["ytick.major.width"] = 0.5
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42


def mm_to_inches(mm: float) -> float:
    return mm / 25.4


def save_dual(fig, output_dir: Path, stem: str):
    svg_path = output_dir / f"{stem}.svg"
    pdf_path = output_dir / f"{stem}.pdf"
    fig.savefig(svg_path, format="svg", bbox_inches="tight")
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    print(f"Saved: {svg_path}")
    print(f"Saved: {pdf_path}")


def format_axes(ax):
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    ax.spines["left"].set_linewidth(0.5)
    ax.spines["bottom"].set_linewidth(0.5)
    ax.spines["left"].set_color("#777777")
    ax.spines["bottom"].set_color("#777777")

    ax.tick_params(axis="both", width=0.5, length=2, colors="#555555")
    ax.grid(axis="y", color="#E6E6E6", linewidth=0.5)
    ax.set_axisbelow(True)


def make_experiment_name(df: pd.DataFrame) -> pd.Series:
    return df["group_label"].astype(str) + "\n" + df["experiment_label"].astype(str)


def prepare_plot_data(all_encounters_df: pd.DataFrame, all_case_cost_df: pd.DataFrame):
    enc_plot = all_encounters_df.copy()
    case_plot = all_case_cost_df.copy()

    if "metadata_missing" in enc_plot.columns:
        enc_plot = enc_plot[~enc_plot["metadata_missing"].fillna(False)].copy()

    enc_plot["experiment_name"] = make_experiment_name(enc_plot)
    case_plot["experiment_name"] = make_experiment_name(case_plot)

    enc_plot["total_tokens_k"] = enc_plot["total_total_tokens"] / 1_000
    enc_plot["requests_per_encounter"] = enc_plot["total_requests"]
    case_plot["five_run_tokens_m"] = case_plot["multi_run_total_tokens"] / 1_000_000

    experiment_order = (
        enc_plot[["group_label", "experiment_label", "experiment_name"]]
        .drop_duplicates()
        .sort_values(["group_label", "experiment_label"])
        ["experiment_name"]
        .tolist()
    )

    palette = {}
    if "color" in enc_plot.columns:
        for _, row in enc_plot[["experiment_name", "color"]].drop_duplicates().iterrows():
            if isinstance(row["color"], str) and row["color"]:
                palette[row["experiment_name"]] = row["color"]

    if not palette:
        palette = {name: "#8FA9C5" for name in experiment_order}

    return enc_plot, case_plot, experiment_order, palette


def plot_single_cost_boxplot(
    data: pd.DataFrame,
    x: str,
    y: str,
    order: list[str],
    palette: dict,
    y_label: str,
    title: str,
    output_dir: Path,
    stem: str,
    width_mm: float = 180,
    height_in: float = 2.9,
):
    setup_style()

    fig, ax = plt.subplots(
        figsize=(mm_to_inches(width_mm), height_in),
        dpi=600,
    )

    sns.boxplot(
        data=data,
        x=x,
        y=y,
        order=order,
        palette=palette,
        width=0.55,
        linewidth=0.6,
        fliersize=1.5,
        saturation=1,
        ax=ax,
    )

    ax.set_title(title, loc="left", fontsize=7, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(y_label)
    format_axes(ax)

    ax.tick_params(axis="x", labelrotation=35)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")

    plt.tight_layout()
    save_dual(fig, output_dir, stem)
    plt.close(fig)


enc_plot, case_plot, experiment_order, palette = prepare_plot_data(
    all_encounters_df,
    all_case_cost_df,
)

plot_single_cost_boxplot(
    data=enc_plot,
    x="experiment_name",
    y="total_tokens_k",
    order=experiment_order,
    palette=palette,
    y_label="Total tokens per encounter (×10³)",
    title="Total tokens per encounter",
    output_dir=FIGURE_OUTPUT_DIR,
    stem="total_tokens_per_encounter",
)

plot_single_cost_boxplot(
    data=enc_plot,
    x="experiment_name",
    y="requests_per_encounter",
    order=experiment_order,
    palette=palette,
    y_label="Requests per encounter",
    title="Requests per encounter",
    output_dir=FIGURE_OUTPUT_DIR,
    stem="requests_per_encounter",
)

plot_single_cost_boxplot(
    data=case_plot,
    x="experiment_name",
    y="five_run_tokens_m",
    order=experiment_order,
    palette=palette,
    y_label="Total tokens per case, five runs (×10⁶)",
    title="Total tokens per case for five-run consistency",
    output_dir=FIGURE_OUTPUT_DIR,
    stem="five_run_tokens_per_case",
)


/tmp/ipykernel_1962012/3257301541.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/total_tokens_per_encounter.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/total_tokens_per_encounter.pdf


/tmp/ipykernel_1962012/3257301541.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/requests_per_encounter.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/requests_per_encounter.pdf


/tmp/ipykernel_1962012/3257301541.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_1962012/3257301541.py:141: UserWarning: Glyph 8310 (\N{SUPERSCRIPT SIX}) missing from font(s) Arial.
  plt.tight_layout()
/tmp/ipykernel_1962012/3257301541.py:41: UserWarning: Glyph 8310 (\N{SUPERSCRIPT SIX}) missing from font(s) Arial.
  fig.savefig(svg_path, format="svg", bbox_inches="tight")


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/five_run_tokens_per_case.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/five_run_tokens_per_case.pdf


/tmp/ipykernel_1962012/3257301541.py:42: UserWarning: Glyph 8310 (\N{SUPERSCRIPT SIX}) missing from font(s) Arial.
  fig.savefig(pdf_path, format="pdf", bbox_inches="tight")


Boxplots summarize computational workload across repeated experiments. a, Total tokens per encounter. b, Model requests per encounter. c, Total tokens per case for the five-run consistency analysis, computed by summing the five independent runs for each case. Boxes show median and interquartile range; whiskers indicate 1.5× IQR.


In [17]:
def make_doctor_case_cost(all_encounters_df: pd.DataFrame) -> pd.DataFrame:
    valid = all_encounters_df.copy()
    if "metadata_missing" in valid.columns:
        valid = valid[~valid["metadata_missing"].fillna(False)].copy()

    group_cols = [
        "group_label",
        "experiment_label",
        "history_dir",
        "note",
        "doctor_model",
        "patient_model",
        "blood_matcher_model",
        "dataset",
        "case_id",
    ]

    rows = []
    for keys, group in valid.groupby(group_cols, dropna=False):
        base = dict(zip(group_cols, keys))
        rows.append({
            **base,
            "n_runs_observed": group["run_id"].nunique(),
            "doctor_multi_run_total_tokens": group["doctor_total_tokens"].sum(),
            "doctor_multi_run_total_requests": group["doctor_requests"].sum(),
            "doctor_single_run_total_tokens_mean": group["doctor_total_tokens"].mean(),
            "doctor_single_run_total_requests_mean": group["doctor_requests"].mean(),
        })

    return pd.DataFrame(rows)


def prepare_doctor_plot_data(all_encounters_df: pd.DataFrame):
    enc_plot = all_encounters_df.copy()
    if "metadata_missing" in enc_plot.columns:
        enc_plot = enc_plot[~enc_plot["metadata_missing"].fillna(False)].copy()

    doctor_case_plot = make_doctor_case_cost(enc_plot)

    enc_plot["experiment_name"] = make_experiment_name(enc_plot)
    doctor_case_plot["experiment_name"] = make_experiment_name(doctor_case_plot)

    enc_plot["doctor_tokens_k"] = enc_plot["doctor_total_tokens"] / 1_000
    enc_plot["doctor_requests_per_encounter"] = enc_plot["doctor_requests"]
    doctor_case_plot["doctor_five_run_tokens_m"] = doctor_case_plot["doctor_multi_run_total_tokens"] / 1_000_000

    experiment_order = (
        enc_plot[["group_label", "experiment_label", "experiment_name"]]
        .drop_duplicates()
        .sort_values(["group_label", "experiment_label"])
        ["experiment_name"]
        .tolist()
    )

    palette = {}
    if "color" in enc_plot.columns:
        for _, row in enc_plot[["experiment_name", "color"]].drop_duplicates().iterrows():
            if isinstance(row["color"], str) and row["color"]:
                palette[row["experiment_name"]] = row["color"]

    if not palette:
        palette = {name: "#8FA9C5" for name in experiment_order}

    return enc_plot, doctor_case_plot, experiment_order, palette


doctor_enc_plot, doctor_case_plot, doctor_experiment_order, doctor_palette = prepare_doctor_plot_data(
    all_encounters_df
)

plot_single_cost_boxplot(
    data=doctor_enc_plot,
    x="experiment_name",
    y="doctor_tokens_k",
    order=doctor_experiment_order,
    palette=doctor_palette,
    y_label="Doctor tokens per encounter (×10³)",
    title="Doctor tokens per encounter",
    output_dir=FIGURE_OUTPUT_DIR,
    stem="doctor_tokens_per_encounter",
)

plot_single_cost_boxplot(
    data=doctor_enc_plot,
    x="experiment_name",
    y="doctor_requests_per_encounter",
    order=doctor_experiment_order,
    palette=doctor_palette,
    y_label="Doctor requests per encounter",
    title="Doctor requests per encounter",
    output_dir=FIGURE_OUTPUT_DIR,
    stem="doctor_requests_per_encounter",
)

plot_single_cost_boxplot(
    data=doctor_case_plot,
    x="experiment_name",
    y="doctor_five_run_tokens_m",
    order=doctor_experiment_order,
    palette=doctor_palette,
    y_label="Doctor tokens per case, five runs (×10⁶)",
    title="Doctor tokens per case for five-run consistency",
    output_dir=FIGURE_OUTPUT_DIR,
    stem="doctor_five_run_tokens_per_case",
)


/tmp/ipykernel_1962012/3257301541.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_tokens_per_encounter.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_tokens_per_encounter.pdf


/tmp/ipykernel_1962012/3257301541.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_requests_per_encounter.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_requests_per_encounter.pdf


/tmp/ipykernel_1962012/3257301541.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_1962012/3257301541.py:141: UserWarning: Glyph 8310 (\N{SUPERSCRIPT SIX}) missing from font(s) Arial.
  plt.tight_layout()
/tmp/ipykernel_1962012/3257301541.py:41: UserWarning: Glyph 8310 (\N{SUPERSCRIPT SIX}) missing from font(s) Arial.
  fig.savefig(svg_path, format="svg", bbox_inches="tight")


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_five_run_tokens_per_case.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_five_run_tokens_per_case.pdf


/tmp/ipykernel_1962012/3257301541.py:42: UserWarning: Glyph 8310 (\N{SUPERSCRIPT SIX}) missing from font(s) Arial.
  fig.savefig(pdf_path, format="pdf", bbox_inches="tight")


In [18]:
def make_doctor_io_case_cost(all_encounters_df: pd.DataFrame) -> pd.DataFrame:
    valid = all_encounters_df.copy()
    if "metadata_missing" in valid.columns:
        valid = valid[~valid["metadata_missing"].fillna(False)].copy()

    group_cols = [
        "group_label",
        "experiment_label",
        "history_dir",
        "note",
        "doctor_model",
        "patient_model",
        "blood_matcher_model",
        "dataset",
        "case_id",
    ]

    rows = []
    for keys, group in valid.groupby(group_cols, dropna=False):
        base = dict(zip(group_cols, keys))
        rows.append({
            **base,
            "n_runs_observed": group["run_id"].nunique(),
            "doctor_multi_run_input_tokens": group["doctor_input_tokens"].sum(),
            "doctor_multi_run_output_tokens": group["doctor_output_tokens"].sum(),
            "doctor_single_run_input_tokens_mean": group["doctor_input_tokens"].mean(),
            "doctor_single_run_output_tokens_mean": group["doctor_output_tokens"].mean(),
        })

    return pd.DataFrame(rows)


def prepare_doctor_io_plot_data(all_encounters_df: pd.DataFrame):
    enc_plot = all_encounters_df.copy()
    if "metadata_missing" in enc_plot.columns:
        enc_plot = enc_plot[~enc_plot["metadata_missing"].fillna(False)].copy()

    doctor_io_case_plot = make_doctor_io_case_cost(enc_plot)

    enc_plot["experiment_name"] = make_experiment_name(enc_plot)
    doctor_io_case_plot["experiment_name"] = make_experiment_name(doctor_io_case_plot)

    enc_plot["doctor_input_tokens_k"] = enc_plot["doctor_input_tokens"] / 1_000
    enc_plot["doctor_output_tokens_k"] = enc_plot["doctor_output_tokens"] / 1_000

    doctor_io_case_plot["doctor_five_run_input_tokens_m"] = (
        doctor_io_case_plot["doctor_multi_run_input_tokens"] / 1_000_000
    )
    doctor_io_case_plot["doctor_five_run_output_tokens_k"] = (
        doctor_io_case_plot["doctor_multi_run_output_tokens"] / 1_000
    )

    experiment_order = (
        enc_plot[["group_label", "experiment_label", "experiment_name"]]
        .drop_duplicates()
        .sort_values(["group_label", "experiment_label"])
        ["experiment_name"]
        .tolist()
    )

    palette = {}
    if "color" in enc_plot.columns:
        for _, row in enc_plot[["experiment_name", "color"]].drop_duplicates().iterrows():
            if isinstance(row["color"], str) and row["color"]:
                palette[row["experiment_name"]] = row["color"]

    if not palette:
        palette = {name: "#8FA9C5" for name in experiment_order}

    return enc_plot, doctor_io_case_plot, experiment_order, palette


doctor_io_enc_plot, doctor_io_case_plot, doctor_io_order, doctor_io_palette = prepare_doctor_io_plot_data(
    all_encounters_df
)

plot_single_cost_boxplot(
    data=doctor_io_enc_plot,
    x="experiment_name",
    y="doctor_input_tokens_k",
    order=doctor_io_order,
    palette=doctor_io_palette,
    y_label="Doctor input tokens per encounter (×10³)",
    title="Doctor input tokens per encounter",
    output_dir=FIGURE_OUTPUT_DIR,
    stem="doctor_input_tokens_per_encounter",
)

plot_single_cost_boxplot(
    data=doctor_io_enc_plot,
    x="experiment_name",
    y="doctor_output_tokens_k",
    order=doctor_io_order,
    palette=doctor_io_palette,
    y_label="Doctor output tokens per encounter (×10³)",
    title="Doctor output tokens per encounter",
    output_dir=FIGURE_OUTPUT_DIR,
    stem="doctor_output_tokens_per_encounter",
)

plot_single_cost_boxplot(
    data=doctor_io_case_plot,
    x="experiment_name",
    y="doctor_five_run_input_tokens_m",
    order=doctor_io_order,
    palette=doctor_io_palette,
    y_label="Doctor input tokens per case, five runs (×10⁶)",
    title="Doctor input tokens per case for five-run consistency",
    output_dir=FIGURE_OUTPUT_DIR,
    stem="doctor_input_tokens_per_case_five_runs",
)

plot_single_cost_boxplot(
    data=doctor_io_case_plot,
    x="experiment_name",
    y="doctor_five_run_output_tokens_k",
    order=doctor_io_order,
    palette=doctor_io_palette,
    y_label="Doctor output tokens per case, five runs (×10³)",
    title="Doctor output tokens per case for five-run consistency",
    output_dir=FIGURE_OUTPUT_DIR,
    stem="doctor_output_tokens_per_case_five_runs",
)

doctor_io_case_path = FIGURE_OUTPUT_DIR / "all_doctor_io_case_level_consistency_cost_raw.csv"
doctor_io_case_plot.to_csv(doctor_io_case_path, index=False)
print(f"Saved doctor input/output case-level cost raw: {doctor_io_case_path}")


/tmp/ipykernel_1962012/3257301541.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_input_tokens_per_encounter.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_input_tokens_per_encounter.pdf


/tmp/ipykernel_1962012/3257301541.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_output_tokens_per_encounter.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_output_tokens_per_encounter.pdf


/tmp/ipykernel_1962012/3257301541.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_1962012/3257301541.py:141: UserWarning: Glyph 8310 (\N{SUPERSCRIPT SIX}) missing from font(s) Arial.
  plt.tight_layout()
/tmp/ipykernel_1962012/3257301541.py:41: UserWarning: Glyph 8310 (\N{SUPERSCRIPT SIX}) missing from font(s) Arial.
  fig.savefig(svg_path, format="svg", bbox_inches="tight")
/tmp/ipykernel_1962012/3257301541.py:42: UserWarning: Glyph 8310 (\N{SUPERSCRIPT SIX}) missing from font(s) Arial.
  fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
/tmp/ipykernel_1962012/3257301541.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_input_tokens_per_case_five_runs.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_input_tokens_per_case_five_runs.pdf
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_output_tokens_per_case_five_runs.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_output_tokens_per_case_five_runs.pdf
Saved doctor input/output case-level cost raw: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/all_doctor_io_case_level_consistency_cost_raw.csv


In [ ]:
def build_experiment_palette_from_groups(groups):
    return {
        f"{group['group_label']}\n{model_cfg['label']}": model_cfg["color"]
        for group in groups
        for model_cfg in group["models"]
        if model_cfg.get("color")
    }


def prepare_single_vs_fiverun_tokens(
    all_encounters_df: pd.DataFrame,
    all_case_cost_df: pd.DataFrame,
    token_scope: str = "total",
):
    """
    token_scope:
      - "total": use total_total_tokens and multi_run_total_tokens
      - "doctor": use doctor_total_tokens and doctor_multi_run_total_tokens
    """
    enc = all_encounters_df.copy()
    if "metadata_missing" in enc.columns:
        enc = enc[~enc["metadata_missing"].fillna(False)].copy()

    case = all_case_cost_df.copy()

    enc["experiment_name"] = make_experiment_name(enc)
    case["experiment_name"] = make_experiment_name(case)

    if token_scope == "total":
        single_col = "total_total_tokens"
        five_col = "multi_run_total_tokens"
        title = "Total tokens: single run vs five-run consistency"
        stem = "total_tokens_single_vs_fiverun"
    elif token_scope == "doctor":
        single_col = "doctor_total_tokens"

        if "doctor_multi_run_total_tokens" not in case.columns:
            doctor_case = make_doctor_case_cost(enc)
            doctor_case["experiment_name"] = make_experiment_name(doctor_case)
            case = doctor_case

        five_col = "doctor_multi_run_total_tokens"
        title = "Doctor tokens: single run vs five-run consistency"
        stem = "doctor_tokens_single_vs_fiverun"
    else:
        raise ValueError("token_scope must be 'total' or 'doctor'.")

    single_df = enc[["experiment_name", "group_label", "experiment_label", single_col]].copy()
    single_df = single_df.rename(columns={single_col: "tokens"})
    single_df["run_setting"] = "Single run per encounter"

    five_df = case[["experiment_name", "group_label", "experiment_label", five_col]].copy()
    five_df = five_df.rename(columns={five_col: "tokens"})
    five_df["run_setting"] = "Five-run per case"

    plot_df = pd.concat([single_df, five_df], ignore_index=True)
    plot_df["tokens_k"] = plot_df["tokens"] / 1_000

    experiment_order = (
        enc[["group_label", "experiment_label", "experiment_name"]]
        .drop_duplicates()
        .sort_values(["group_label", "experiment_label"])
        ["experiment_name"]
        .tolist()
    )

    experiment_palette = build_experiment_palette_from_groups(GROUPS)

    # return plot_df, experiment_order, title, stem
    return plot_df, experiment_order, experiment_palette, title, stem


def plot_single_vs_fiverun_tokens(
    all_encounters_df: pd.DataFrame,
    all_case_cost_df: pd.DataFrame,
    output_dir: Path,
    token_scope: str = "total",
):
    setup_style()

    plot_df, experiment_order, experiment_palette, title, stem = prepare_single_vs_fiverun_tokens(
        all_encounters_df,
        all_case_cost_df,
        token_scope=token_scope,
    )

    fig, ax = plt.subplots(
        figsize=(mm_to_inches(180), 3.1),
        dpi=600,
    )

    hue_order = ["Single run per encounter", "Five-run per case"]
    hue_palette = {
        "Single run per encounter": "#9FB7CC",
        "Five-run per case": "#D07B5F",
    }

    sns.boxplot(
        data=plot_df,
        x="experiment_name",
        y="tokens_k",
        hue="run_setting",
        order=experiment_order,
        hue_order=hue_order,
        palette=hue_palette,
        width=0.65,
        linewidth=0.6,
        fliersize=1.5,
        saturation=1,
        ax=ax,
    )


    ax.set_title(title, loc="left", fontsize=7, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Tokens (×10³)")
    format_axes(ax)

    ax.tick_params(axis="x", labelrotation=35)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")

    ax.legend(
        title="",
        frameon=False,
        loc="upper left",
        fontsize=5,
    )

    plt.tight_layout()
    save_dual(fig, output_dir, stem)
    plt.close(fig)

    csv_path = output_dir / f"{stem}_plot_data.csv"
    plot_df.to_csv(csv_path, index=False)
    print(f"Saved plot data: {csv_path}")


plot_single_vs_fiverun_tokens(
    all_encounters_df=all_encounters_df,
    all_case_cost_df=all_case_cost_df,
    output_dir=FIGURE_OUTPUT_DIR,
    token_scope="total",
)

plot_single_vs_fiverun_tokens(
    all_encounters_df=all_encounters_df,
    all_case_cost_df=all_case_cost_df,
    output_dir=FIGURE_OUTPUT_DIR,
    token_scope="doctor",
)


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/total_tokens_single_vs_fiverun.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/total_tokens_single_vs_fiverun.pdf
Saved plot data: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/total_tokens_single_vs_fiverun_plot_data.csv
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_tokens_single_vs_fiverun.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_tokens_single_vs_fiverun.pdf
Saved plot data: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_tokens_single_vs_fiverun_plot_data.csv


In [ ]:
import matplotlib.patches as mpatches


def build_experiment_palette_from_groups(groups: list[dict]) -> dict:
    """Map 'group_label\\nexperiment_label' to color from GROUPS."""
    palette = {}
    for group in groups:
        group_label = group["group_label"]
        for model_cfg in group["models"]:
            exp_name = f"{group_label}\n{model_cfg['label']}"
            if model_cfg.get("color"):
                palette[exp_name] = model_cfg["color"]
    return palette


def build_experiment_order_from_groups(groups: list[dict], available_names: set[str]) -> list[str]:
    """Use GROUPS order, keeping only experiments present in the data."""
    order = []
    for group in groups:
        group_label = group["group_label"]
        for model_cfg in group["models"]:
            exp_name = f"{group_label}\n{model_cfg['label']}"
            if exp_name in available_names:
                order.append(exp_name)
    return order


def prepare_single_vs_fiverun_tokens(
    all_encounters_df: pd.DataFrame,
    all_case_cost_df: pd.DataFrame,
    token_scope: str = "total",
):
    """
    token_scope:
      - "total": use total_total_tokens and multi_run_total_tokens
      - "doctor": use doctor_total_tokens and doctor_multi_run_total_tokens
    """
    enc = all_encounters_df.copy()
    if "metadata_missing" in enc.columns:
        enc = enc[~enc["metadata_missing"].fillna(False)].copy()

    case = all_case_cost_df.copy()

    enc["experiment_name"] = make_experiment_name(enc)
    case["experiment_name"] = make_experiment_name(case)

    if token_scope == "total":
        single_col = "total_total_tokens"
        five_col = "multi_run_total_tokens"
        title = "Total tokens: single run vs five-run"
        stem = "total_tokens_single_vs_fiverun"
    elif token_scope == "doctor":
        single_col = "doctor_total_tokens"

        if "doctor_multi_run_total_tokens" not in case.columns:
            doctor_case = make_doctor_case_cost(enc)
            doctor_case["experiment_name"] = make_experiment_name(doctor_case)
            case = doctor_case

        five_col = "doctor_multi_run_total_tokens"
        title = "Doctor tokens: single run vs five-run"
        stem = "doctor_tokens_single_vs_fiverun"
    else:
        raise ValueError("token_scope must be 'total' or 'doctor'.")

    single_df = enc[["experiment_name", "group_label", "experiment_label", single_col]].copy()
    single_df = single_df.rename(columns={single_col: "tokens"})
    single_df["run_setting"] = "Single run per encounter"

    five_df = case[["experiment_name", "group_label", "experiment_label", five_col]].copy()
    five_df = five_df.rename(columns={five_col: "tokens"})
    five_df["run_setting"] = "Five-run per case"

    plot_df = pd.concat([single_df, five_df], ignore_index=True)
    plot_df["tokens_k"] = plot_df["tokens"] / 1_000

    available_names = set(plot_df["experiment_name"].dropna().unique())
    experiment_order = build_experiment_order_from_groups(GROUPS, available_names)

    # Fallback: include any experiment not listed in GROUPS.
    remaining = [name for name in available_names if name not in experiment_order]
    experiment_order.extend(sorted(remaining))

    experiment_palette = build_experiment_palette_from_groups(GROUPS)
    for name in remaining:
        experiment_palette.setdefault(name, "#8FA9C5")

    return plot_df, experiment_order, experiment_palette, title, stem


def plot_single_vs_fiverun_tokens(
    all_encounters_df: pd.DataFrame,
    all_case_cost_df: pd.DataFrame,
    output_dir: Path,
    token_scope: str = "total",
):
    setup_style()
    plt.rcParams["hatch.linewidth"] = 0.15

    plot_df, experiment_order, experiment_palette, title, stem = prepare_single_vs_fiverun_tokens(
        all_encounters_df,
        all_case_cost_df,
        token_scope=token_scope,
    )

    fig, ax = plt.subplots(
        figsize=(mm_to_inches(180), 1.6),
        dpi=600,
    )

    run_settings = ["Single run per encounter", "Five-run per case"]
    offsets = [-0.17, 0.17]
    box_width = 0.26

    for exp_idx, exp_name in enumerate(experiment_order):
        color = experiment_palette.get(exp_name, "#8FA9C5")

        for setting, offset in zip(run_settings, offsets):
            values = (
                plot_df[
                    (plot_df["experiment_name"] == exp_name)
                    & (plot_df["run_setting"] == setting)
                ]["tokens_k"]
                .dropna()
                .to_numpy()
            )

            if len(values) == 0:
                continue

            bp = ax.boxplot(
                [values],
                positions=[exp_idx + offset],
                widths=box_width,
                patch_artist=True,
                showfliers=False, #True,
                flierprops=dict(
                    marker="o",
                    markersize=1.5,
                    markerfacecolor=color,
                    markeredgecolor="none",
                    alpha=0.45,
                ),
                boxprops=dict(
                    facecolor=color,
                    edgecolor="black",
                    linewidth=0.45,
                ),
                whiskerprops=dict(color="black", linewidth=0.45),
                capprops=dict(color="black", linewidth=0.45),
                medianprops=dict(color="black", linewidth=0.6),
            )

            if setting == "Five-run per case":
                bp["boxes"][0].set_hatch("////")

    ax.set_title(title, loc="left", fontsize=7, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Tokens (×10³)")

    ax.set_xticks(range(len(experiment_order)))
    ax.set_xticklabels(experiment_order, rotation=35, ha="right")

    format_axes(ax)
    ax.grid(False)

    legend_handles = [
        mpatches.Patch(
            facecolor="white",
            edgecolor="black",
            linewidth=0.45,
            label="Single run per encounter",
        ),
        mpatches.Patch(
            facecolor="white",
            edgecolor="black",
            linewidth=0.45,
            hatch="////",
            label="Five-run per case",
        ),
    ]

    ax.legend(
        handles=legend_handles,
        title="",
        frameon=False,
        loc="upper left",
        fontsize=5,
    )

    plt.tight_layout()
    save_dual(fig, output_dir, stem)
    plt.close(fig)

    csv_path = output_dir / f"{stem}_plot_data.csv"
    plot_df.to_csv(csv_path, index=False)
    print(f"Saved plot data: {csv_path}")


plot_single_vs_fiverun_tokens(
    all_encounters_df=all_encounters_df,
    all_case_cost_df=all_case_cost_df,
    output_dir=FIGURE_OUTPUT_DIR,
    token_scope="total",
)

plot_single_vs_fiverun_tokens(
    all_encounters_df=all_encounters_df,
    all_case_cost_df=all_case_cost_df,
    output_dir=FIGURE_OUTPUT_DIR,
    token_scope="doctor",
)


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/total_tokens_single_vs_fiverun.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/total_tokens_single_vs_fiverun.pdf
Saved plot data: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/total_tokens_single_vs_fiverun_plot_data.csv
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_tokens_single_vs_fiverun.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_tokens_single_vs_fiverun.pdf
Saved plot data: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/doctor_tokens_single_vs_fiverun_plot_data.csv


In [ ]:
def prepare_latency_plot_data(
    all_encounters_df: pd.DataFrame,
    all_case_cost_df: pd.DataFrame,
):
    enc = all_encounters_df.copy()
    if "metadata_missing" in enc.columns:
        enc = enc[~enc["metadata_missing"].fillna(False)].copy()

    case = all_case_cost_df.copy()

    enc["experiment_name"] = make_experiment_name(enc)
    case["experiment_name"] = make_experiment_name(case)

    enc["wall_clock_minutes"] = enc["wall_clock_seconds"] / 60.0
    enc["total_tokens_k"] = enc["total_total_tokens"] / 1_000

    case["single_run_minutes_case_mean"] = case["single_run_wall_clock_seconds_mean"] / 60.0
    case["serial_5run_minutes"] = case["serial_multi_run_wall_clock_seconds"] / 60.0
    case["parallel_5run_lower_bound_minutes"] = (
        case["parallel_multi_run_wall_clock_lower_bound_seconds"] / 60.0
    )

    experiment_order = (
        enc[["group_label", "experiment_label", "experiment_name"]]
        .drop_duplicates()
        .sort_values(["group_label", "experiment_label"])
        ["experiment_name"]
        .tolist()
    )

    palette = {}
    if "color" in enc.columns:
        for _, row in enc[["experiment_name", "color"]].drop_duplicates().iterrows():
            if isinstance(row["color"], str) and row["color"]:
                palette[row["experiment_name"]] = row["color"]

    if not palette:
        palette = {name: "#8FA9C5" for name in experiment_order}

    return enc, case, experiment_order, palette


def plot_wall_clock_minutes_per_encounter(
    all_encounters_df: pd.DataFrame,
    all_case_cost_df: pd.DataFrame,
    output_dir: Path,
):
    setup_style()

    enc, _, experiment_order, palette = prepare_latency_plot_data(
        all_encounters_df,
        all_case_cost_df,
    )

    fig, ax = plt.subplots(figsize=(mm_to_inches(180), 3.0), dpi=600)

    sns.boxplot(
        data=enc,
        x="experiment_name",
        y="wall_clock_minutes",
        order=experiment_order,
        palette=palette,
        width=0.55,
        linewidth=0.6,
        fliersize=1.5,
        saturation=1,
        ax=ax,
    )

    ax.set_title("Wall-clock time per encounter", loc="left", fontsize=7, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Wall-clock time (min)")
    format_axes(ax)
    ax.grid(False)

    ax.tick_params(axis="x", labelrotation=35)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")

    plt.tight_layout()
    save_dual(fig, output_dir, "latency_wall_clock_minutes_per_encounter")
    plt.close(fig)


def plot_single_vs_fiverun_latency(
    all_encounters_df: pd.DataFrame,
    all_case_cost_df: pd.DataFrame,
    output_dir: Path,
):
    setup_style()

    enc, case, experiment_order, _ = prepare_latency_plot_data(
        all_encounters_df,
        all_case_cost_df,
    )

    single_df = enc[["experiment_name", "group_label", "experiment_label", "wall_clock_minutes"]].copy()
    single_df = single_df.rename(columns={"wall_clock_minutes": "minutes"})
    single_df["run_setting"] = "Single run per encounter"

    serial_df = case[["experiment_name", "group_label", "experiment_label", "serial_5run_minutes"]].copy()
    serial_df = serial_df.rename(columns={"serial_5run_minutes": "minutes"})
    serial_df["run_setting"] = "Five-run serial per case"

    parallel_df = case[["experiment_name", "group_label", "experiment_label", "parallel_5run_lower_bound_minutes"]].copy()
    parallel_df = parallel_df.rename(columns={"parallel_5run_lower_bound_minutes": "minutes"})
    parallel_df["run_setting"] = "Five-run parallel lower bound"

    plot_df = pd.concat([single_df, serial_df, parallel_df], ignore_index=True)

    fig, ax = plt.subplots(figsize=(mm_to_inches(180), 3.2), dpi=600)

    hue_order = [
        "Single run per encounter",
        "Five-run serial per case",
        "Five-run parallel lower bound",
    ]
    hue_palette = {
        "Single run per encounter": "#9FB7CC",
        "Five-run serial per case": "#D07B5F",
        "Five-run parallel lower bound": "#7BAF87",
    }

    sns.boxplot(
        data=plot_df,
        x="experiment_name",
        y="minutes",
        hue="run_setting",
        order=experiment_order,
        hue_order=hue_order,
        palette=hue_palette,
        width=0.70,
        linewidth=0.6,
        fliersize=1.5,
        saturation=1,
        ax=ax,
    )

    ax.set_title("Observed latency: single run vs five-run consistency", loc="left", fontsize=7, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Wall-clock time (min)")
    format_axes(ax)

    ax.tick_params(axis="x", labelrotation=35)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")

    ax.legend(title="", frameon=False, loc="upper left", fontsize=5)

    plt.tight_layout()
    save_dual(fig, output_dir, "latency_single_vs_fiverun_consistency")
    plt.close(fig)

    csv_path = output_dir / "latency_single_vs_fiverun_consistency_plot_data.csv"
    plot_df.to_csv(csv_path, index=False)
    print(f"Saved plot data: {csv_path}")


def plot_tokens_vs_wall_clock_scatter(
    all_encounters_df: pd.DataFrame,
    all_case_cost_df: pd.DataFrame,
    output_dir: Path,
):
    setup_style()

    enc, _, experiment_order, palette = prepare_latency_plot_data(
        all_encounters_df,
        all_case_cost_df,
    )

    fig, ax = plt.subplots(figsize=(mm_to_inches(180), 3.2), dpi=600)

    sns.scatterplot(
        data=enc,
        x="total_tokens_k",
        y="wall_clock_minutes",
        hue="experiment_name",
        hue_order=experiment_order,
        palette=palette,
        s=8,
        alpha=0.35,
        linewidth=0,
        ax=ax,
    )

    ax.set_title("Total tokens vs wall-clock time", loc="left", fontsize=7, fontweight="bold")
    ax.set_xlabel("Total tokens per encounter (×10³)")
    ax.set_ylabel("Wall-clock time (min)")
    format_axes(ax)

    ax.legend(
        title="",
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0,
        fontsize=5,
    )

    plt.tight_layout()
    save_dual(fig, output_dir, "latency_total_tokens_vs_wall_clock_scatter")
    plt.close(fig)


plot_wall_clock_minutes_per_encounter(
    all_encounters_df=all_encounters_df,
    all_case_cost_df=all_case_cost_df,
    output_dir=FIGURE_OUTPUT_DIR,
)

plot_single_vs_fiverun_latency(
    all_encounters_df=all_encounters_df,
    all_case_cost_df=all_case_cost_df,
    output_dir=FIGURE_OUTPUT_DIR,
)

plot_tokens_vs_wall_clock_scatter(
    all_encounters_df=all_encounters_df,
    all_case_cost_df=all_case_cost_df,
    output_dir=FIGURE_OUTPUT_DIR,
)


/tmp/ipykernel_1962012/2744618317.py:57: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/latency_wall_clock_minutes_per_encounter.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/latency_wall_clock_minutes_per_encounter.pdf
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/latency_single_vs_fiverun_consistency.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/latency_single_vs_fiverun_consistency.pdf
Saved plot data: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/latency_single_vs_fiverun_consistency_plot_data.csv
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/latency_total_tokens_vs_wall_clock_scatter.svg
Saved: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/lat

In [3]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

LATENCY_ROOT = Path("/mnt/bulk-uranus/lizhang/LiWorkSpace/data/history/latency_test")
OUT_DIR = Path("/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/latency_subset")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Change this if your full benchmark case count differs.
FULL_BENCHMARK_CASES_PER_MODEL = 551


def parse_ts(x):
    return datetime.fromisoformat(x) if x else None


def event_source(event):
    """Classify timestamped events for doctor-latency reconstruction."""
    if event.get("type") in {"run_metadata", "logprob_analysis"}:
        return "ignore"
    if event.get("role") == "Patient":
        return "patient"
    if event.get("role") == "Doctor":
        return "doctor"
    if event.get("type") == "action" and event.get("role") == "System":
        # System action events such as request_blood_test / finish are doctor-generated tool calls.
        return "doctor"
    return "other"


def estimate_doctor_wall_clock_seconds(events):
    """
    Estimate doctor-side wall-clock latency by summing intervals from the
    previous non-doctor event to each doctor-generated event group.

    This avoids overcounting multiple tool-call events emitted at nearly the
    same timestamp within one doctor turn.
    """
    timestamped = []
    for event in events:
        source = event_source(event)
        ts = parse_ts(event.get("timestamp"))
        if source == "ignore" or ts is None:
            continue
        timestamped.append((ts, source))

    if not timestamped:
        return np.nan

    timestamped.sort(key=lambda x: x[0])

    groups = []
    current_source = timestamped[0][1]
    first_ts = timestamped[0][0]
    last_ts = timestamped[0][0]

    for ts, source in timestamped[1:]:
        if source == current_source:
            last_ts = ts
        else:
            groups.append((current_source, first_ts, last_ts))
            current_source = source
            first_ts = ts
            last_ts = ts

    groups.append((current_source, first_ts, last_ts))

    doctor_seconds = 0.0
    previous_last_ts = None

    for source, group_first_ts, group_last_ts in groups:
        if source == "doctor" and previous_last_ts is not None:
            doctor_seconds += (group_first_ts - previous_last_ts).total_seconds()
        previous_last_ts = group_last_ts

    return doctor_seconds


def parse_latency_json(path):
    rel = path.relative_to(LATENCY_ROOT)
    model, run_name, disease = rel.parts[:3]

    run_idx = np.nan
    if run_name.startswith("run_") and run_name.split("_")[-1].isdigit():
        run_idx = int(run_name.split("_")[-1])

    stem = path.stem
    case_id = stem.replace(f"{disease}_", "").split("_conversation_")[0]
    case_key = f"{disease}_{case_id}"

    with open(path, "r") as f:
        events = json.load(f)

    metadata = next(
        (x.get("metadata", {}) for x in events if x.get("type") == "run_metadata"),
        {},
    )

    token_usage = metadata.get("token_usage", {})
    doctor_usage = token_usage.get("doctor", {})
    total_usage = token_usage.get("total", {})

    doctor_wall_clock = estimate_doctor_wall_clock_seconds(events)
    doctor_total_tokens = doctor_usage.get("total_tokens", np.nan)

    return {
        "model": model,
        "run": run_name,
        "run_idx": run_idx,
        "disease": disease,
        "case_id": case_id,
        "case_key": case_key,
        "file": str(path),
        "case_wall_clock_seconds": metadata.get("wall_clock_seconds", np.nan),
        "doctor_wall_clock_seconds": doctor_wall_clock,
        "doctor_requests": doctor_usage.get("requests", np.nan),
        "doctor_input_tokens": doctor_usage.get("input_tokens", np.nan),
        "doctor_output_tokens": doctor_usage.get("output_tokens", np.nan),
        "doctor_total_tokens": doctor_total_tokens,
        "total_tokens": total_usage.get("total_tokens", np.nan),
        "doctor_tokens_per_second": doctor_total_tokens / doctor_wall_clock
        if doctor_wall_clock and doctor_wall_clock > 0
        else np.nan,
    }


def summarize(df, group_cols, value_col):
    return (
        df.groupby(group_cols, dropna=False)[value_col]
        .agg(
            n="count",
            mean="mean",
            sd="std",
            median="median",
            q1=lambda x: x.quantile(0.25),
            q3=lambda x: x.quantile(0.75),
            min="min",
            max="max",
        )
        .reset_index()
    )


# 1. Parse all latency JSON files.
records = [parse_latency_json(p) for p in LATENCY_ROOT.glob("*/*/*/*.json")]
latency_df = pd.DataFrame(records)

latency_df.to_csv(OUT_DIR / "latency_per_encounter_subset.csv", index=False)


# 2. Single-run latency summaries.
single_model_summary = summarize(
    latency_df,
    ["model"],
    "doctor_wall_clock_seconds",
).rename(
    columns={
        "mean": "single_run_latency_mean_s",
        "sd": "single_run_latency_sd_s",
        "median": "single_run_latency_median_s",
        "q1": "single_run_latency_q1_s",
        "q3": "single_run_latency_q3_s",
        "min": "single_run_latency_min_s",
        "max": "single_run_latency_max_s",
    }
)

single_disease_summary = summarize(
    latency_df,
    ["model", "disease"],
    "doctor_wall_clock_seconds",
)

single_model_summary.to_csv(
    OUT_DIR / "single_run_latency_summary_by_model.csv",
    index=False,
)
single_disease_summary.to_csv(
    OUT_DIR / "single_run_latency_summary_by_model_disease.csv",
    index=False,
)


# 3. Five-run latency: measured if five runs exist, otherwise estimated as 5 x observed mean.
five_run_rows = []

for (model, disease, case_key), group in latency_df.sort_values("run_idx").groupby(
    ["model", "disease", "case_key"],
    dropna=False,
):
    first_five = group.head(5)
    n_observed_runs = len(first_five)

    if n_observed_runs >= 5:
        method = "measured_sum_of_5_runs"
        doctor_five_run_s = first_five["doctor_wall_clock_seconds"].sum()
        case_five_run_s = first_five["case_wall_clock_seconds"].sum()
        doctor_five_run_tokens = first_five["doctor_total_tokens"].sum()
    else:
        method = f"estimated_from_{n_observed_runs}_observed_run_mean_x5"
        doctor_five_run_s = first_five["doctor_wall_clock_seconds"].mean() * 5
        case_five_run_s = first_five["case_wall_clock_seconds"].mean() * 5
        doctor_five_run_tokens = first_five["doctor_total_tokens"].mean() * 5

    five_run_rows.append(
        {
            "model": model,
            "disease": disease,
            "case_key": case_key,
            "n_observed_runs": n_observed_runs,
            "five_run_method": method,
            "five_run_latency_seconds": doctor_five_run_s,
            "five_run_case_wall_clock_seconds": case_five_run_s,
            "five_run_doctor_total_tokens": doctor_five_run_tokens,
        }
    )

five_run_df = pd.DataFrame(five_run_rows)

five_model_summary = summarize(
    five_run_df,
    ["model", "five_run_method"],
    "five_run_latency_seconds",
).rename(
    columns={
        "mean": "five_run_latency_mean_s",
        "sd": "five_run_latency_sd_s",
        "median": "five_run_latency_median_s",
        "q1": "five_run_latency_q1_s",
        "q3": "five_run_latency_q3_s",
        "min": "five_run_latency_min_s",
        "max": "five_run_latency_max_s",
    }
)

five_run_df.to_csv(OUT_DIR / "five_run_latency_per_case.csv", index=False)
five_model_summary.to_csv(
    OUT_DIR / "five_run_latency_summary_by_model.csv",
    index=False,
)


# 4. Token/s sensitivity analysis.
token_speed_summary = summarize(
    latency_df,
    ["model"],
    "doctor_tokens_per_second",
).rename(
    columns={
        "mean": "doctor_tokens_per_second_mean",
        "sd": "doctor_tokens_per_second_sd",
        "median": "doctor_tokens_per_second_median",
        "q1": "doctor_tokens_per_second_q1",
        "q3": "doctor_tokens_per_second_q3",
    }
)

token_speed_summary.to_csv(
    OUT_DIR / "doctor_token_throughput_sensitivity_by_model.csv",
    index=False,
)


# 5. Estimated full-benchmark serial runtime.
runtime_summary = single_model_summary.merge(
    five_model_summary,
    on="model",
    how="left",
)

runtime_summary["estimated_single_run_full_benchmark_hours"] = (
    runtime_summary["single_run_latency_mean_s"]
    * FULL_BENCHMARK_CASES_PER_MODEL
    / 3600
)

runtime_summary["estimated_five_run_full_benchmark_hours"] = (
    runtime_summary["five_run_latency_mean_s"]
    * FULL_BENCHMARK_CASES_PER_MODEL
    / 3600
)

runtime_summary.to_csv(
    OUT_DIR / "estimated_full_benchmark_serial_runtime_by_model.csv",
    index=False,
)

print(f"Parsed {len(latency_df)} encounter runs.")
print(f"Models: {latency_df['model'].nunique()}")
print(f"Saved outputs to: {OUT_DIR}")
display(runtime_summary)


Parsed 140 encounter runs.
Models: 4
Saved outputs to: /mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/computational_cost_group_summary/latency_subset


,model,n_x,single_run_latency_mean_s,single_run_latency_sd_s,single_run_latency_median_s,single_run_latency_q1_s,single_run_latency_q3_s,single_run_latency_min_s,single_run_latency_max_s,five_run_method,n_y,five_run_latency_mean_s,five_run_latency_sd_s,five_run_latency_median_s,five_run_latency_q1_s,five_run_latency_q3_s,five_run_latency_min_s,five_run_latency_max_s,estimated_single_run_full_benchmark_hours,estimated_five_run_full_benchmark_hours
0,GLM-4.5-Air-FP8,35,40.789397,10.727862,38.440494,32.719006,47.348222,27.061868,78.037695,estimated_from_1_observed_run_mean_x5,35,203.946987,53.639312,192.202470,163.595030,236.741108,135.309340,390.188475,6.243044,31.215219
1,GLM-5-FP8,35,117.324069,30.650813,112.234531,94.048983,140.802764,69.758011,205.141628,estimated_from_1_observed_run_mean_x5,35,586.620347,153.254067,561.172655,470.244915,704.013820,348.790055,1025.708140,17.957101,89.785503
2,GPT-OSS-120B,35,152.312618,80.814220,125.569880,100.158733,189.241869,59.797187,373.181370,estimated_from_1_observed_run_mean_x5,35,761.563092,404.071102,627.849400,500.793667,946.209345,298.985935,1865.906850,23.312292,116.561462
3,Qwen3.5-397B-A17B-FP8_1,35,117.168481,60.743066,96.032421,70.334238,149.486597,55.281862,295.257336,estimated_from_1_observed_run_mean_x5,35,585.842403,303.715330,480.162105,351.671187,747.432985,276.409310,1476.286680,17.933287,89.666435
